# Revenue integrity findings (Part 3)

Structured investigation for `DESIGN.md` Section 3. Each finding includes detection SQL, quantified results, business impact, and pipeline handling.

**Prerequisites:** ClickHouse on `localhost:8123`, Dagster ingestion materialised, `pip install -r notebooks/requirements.txt`

**Related:** exploratory charts live in `revenue_investigation.ipynb`; this notebook is the evidence pack for write-up.

In [1]:
import clickhouse_connect
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

client = clickhouse_connect.get_client(host="localhost", port=8123)

def ch(sql: str) -> pd.DataFrame:
    return client.query_df(sql)

def show_finding(title: str, what: str, why: str, handled: str) -> None:
    print("=" * 72)
    print(title)
    print("=" * 72)
    print("WHAT:", what)
    print("WHY IT MATTERS:", why)
    print("HOW HANDLED:", handled)


# ClickHouse 24.8 tips: avoid GROUP BY column aliases; do not alias aggregates
# as revenue_usd when the table has revenue_usd; compute ratios in pandas.

## Finding 1 — Publisher 11 × Campaign 1051 revenue spike

**Hypothesis:** A small set of impressions carries most platform revenue at implausible CPM vs peers on the same campaign.

In [4]:
show_finding(
    "Finding 1: Publisher 11 / Campaign 1051 CPM spike",
    what="410 events (0.22% of paid rows) have revenue_usd > $1, totalling $1,455.27 — 77.4% of all revenue ($1,879.10). "
         "Publisher 11 on campaign 1051: CPM $1,273 vs $2.38 for all other publishers. "
         "Spike starts 2026-03-20 UTC; campaign 1051 total on pub 11 ≈ $1,489.49.",
    why="Daily and campaign reporting is dominated by ~400 rows. Finance reconciliation and publisher payouts would be wrong without review.",
    handled="Documented in mart; not excluded in pipeline. Production: quarantine flag, SSP ticket, cap or exclude until confirmed.",
)

spike_summary = ch("""
SELECT
    countIf(toFloat64(revenue_usd) > 1) AS outlier_events,
    round(sumIf(toFloat64(revenue_usd), toFloat64(revenue_usd) > 1), 6) AS outlier_revenue_usd,
    round(sum(toFloat64(revenue_usd)), 6) AS total_revenue_usd
FROM raw.ad_events
WHERE revenue_usd IS NOT NULL
""")
spike_summary["pct_of_total_revenue"] = (
    spike_summary["outlier_revenue_usd"] / spike_summary["total_revenue_usd"] * 100
).round(2)
spike_summary

cpm_by_cohort = ch("""
SELECT
    if(publisher_id = 11, 'publisher_11_retrogames', 'all_other_publishers') AS cohort,
    count() AS impressions,
    sum(toFloat64(revenue_usd)) AS total_revenue_usd
FROM raw.ad_events
WHERE campaign_id = 1051 AND event_type = 'impression'
GROUP BY if(publisher_id = 11, 'publisher_11_retrogames', 'all_other_publishers')
ORDER BY cohort
""")
cpm_by_cohort["total_revenue_usd"] = cpm_by_cohort["total_revenue_usd"].round(6)
cpm_by_cohort["cpm"] = (
    cpm_by_cohort["total_revenue_usd"] * 1000 / cpm_by_cohort["impressions"]
).round(2)
cpm_by_cohort

spike_daily = ch("""
SELECT
    toDate(event_timestamp) AS event_date_utc,
    count() AS events,
    countIf(toFloat64(revenue_usd) > 1) AS high_revenue_events,
    round(sum(toFloat64(revenue_usd)), 6) AS total_revenue_usd
FROM raw.ad_events
WHERE campaign_id = 1051 AND publisher_id = 11
GROUP BY toDate(event_timestamp)
ORDER BY event_date_utc
""")
spike_daily

Finding 1: Publisher 11 / Campaign 1051 CPM spike
WHAT: 410 events (0.22% of paid rows) have revenue_usd > $1, totalling $1,455.27 — 77.4% of all revenue ($1,879.10). Publisher 11 on campaign 1051: CPM $1,273 vs $2.38 for all other publishers. Spike starts 2026-03-20 UTC; campaign 1051 total on pub 11 ≈ $1,489.49.
WHY IT MATTERS: Daily and campaign reporting is dominated by ~400 rows. Finance reconciliation and publisher payouts would be wrong without review.
HOW HANDLED: Documented in mart; not excluded in pipeline. Production: quarantine flag, SSP ticket, cap or exclude until confirmed.


,event_date_utc,events,high_revenue_events,total_revenue_usd
0,2026-03-15,133,0,3.943587
1,2026-03-16,140,0,4.469430
2,2026-03-17,157,0,4.952640
3,2026-03-18,128,0,3.917502
4,2026-03-19,146,0,4.774192
5,2026-03-20,143,131,464.705200
6,2026-03-21,132,111,401.279700
7,2026-03-22,77,67,233.178719
8,2026-03-23,66,54,196.464073
9,2026-03-24,60,47,171.801029


## Finding 2 — Filled impressions with zero revenue

**Hypothesis:** `is_filled = 1` should imply billable delivery; zero/null revenue breaks fill-rate vs revenue consistency.

In [5]:
show_finding(
    "Finding 2: Filled events with $0 revenue",
    what="12,324 events (6.5% of 189,972) have is_filled = 1 and revenue_usd NULL or 0. $0 direct revenue impact but they inflate filled inventory.",
    why="Fill rate overstates monetisable inventory; publisher reporting may count serves that never paid.",
    handled="Included in fct fill_rate denominator (filled/total events). Production: separate metric billable_impressions excluding zero-revenue fills.",
)

filled_zero = ch("""
SELECT count() AS filled_zero_revenue_events
FROM raw.ad_events
WHERE is_filled = 1
  AND (revenue_usd IS NULL OR toFloat64(revenue_usd) = 0)
""")
total_events = ch("SELECT count() AS total_events FROM raw.ad_events").iloc[0, 0]
filled_zero["pct_of_all_events"] = round(
    filled_zero["filled_zero_revenue_events"].iloc[0] / total_events * 100, 2
)
filled_zero

Finding 2: Filled events with $0 revenue
WHAT: 12,324 events (6.5% of 189,972) have is_filled = 1 and revenue_usd NULL or 0. $0 direct revenue impact but they inflate filled inventory.
WHY IT MATTERS: Fill rate overstates monetisable inventory; publisher reporting may count serves that never paid.
HOW HANDLED: Included in fct fill_rate denominator (filled/total events). Production: separate metric billable_impressions excluding zero-revenue fills.


,filled_zero_revenue_events,pct_of_all_events
0,12324,6.490000


## Finding 3 — Dimension join fan-out without FINAL

**Hypothesis:** ReplacingMergeTree dimension tables duplicate keys until merge; naive joins double-count facts.

In [6]:
show_finding(
    "Finding 3: Join fan-out on raw dimensions",
    what="189,972 fact rows become 379,944 when joining raw.publishers without FINAL (+100%). "
         "With FINAL: 189,972 rows. Any SUM(revenue) after naive join over-counts by 2× on publishers.",
    why="Aggregate revenue integrity fails silently in ad-hoc SQL and broken dbt models.",
    handled="stg_* dimensions use FROM raw.* FINAL; documented in pipeline DESIGN.",
)

fanout = ch("""
SELECT
    (SELECT count() FROM raw.ad_events) AS fact_rows,
    (SELECT count()
     FROM raw.ad_events AS e
     INNER JOIN raw.publishers AS p ON e.publisher_id = p.publisher_id) AS join_without_final,
    (SELECT count()
     FROM raw.ad_events AS e
     INNER JOIN (SELECT * FROM raw.publishers FINAL) AS p ON e.publisher_id = p.publisher_id) AS join_with_final
""")
fanout

Finding 3: Join fan-out on raw dimensions
WHAT: 189,972 fact rows become 379,944 when joining raw.publishers without FINAL (+100%). With FINAL: 189,972 rows. Any SUM(revenue) after naive join over-counts by 2× on publishers.
WHY IT MATTERS: Aggregate revenue integrity fails silently in ad-hoc SQL and broken dbt models.
HOW HANDLED: stg_* dimensions use FROM raw.* FINAL; documented in pipeline DESIGN.


,fact_rows,join_without_final,join_with_final
0,189972,379944,189972


## Finding 4 — Campaign 1017 flight date / status (SCD redelivery)

**Hypothesis:** Dimension redelivery changes flight end; events look out-of-flight under stale attributes.

In [7]:
show_finding(
    "Finding 4: Campaign 1017 flight mismatch",
    what="Initial export: status completed, end 2026-03-05. Redelivery (FINAL): active, end 2026-03-20. "
         "224 events between 2026-03-06 and 2026-03-20 ($0.48 revenue) are in-flight under redelivery only.",
    why="Flight filters and budget pacing depend on dimension version; wrong version misclassifies valid traffic.",
    handled="Dagster loads redelivery overlay; dbt uses FINAL (SCD Type 1 latest). Production: SCD2 if audit trail required.",
)

campaign_versions = ch("""
SELECT campaign_id, campaign_status, campaign_end_date, _loaded_at
FROM raw.campaigns
WHERE campaign_id = 1017
ORDER BY _loaded_at
""")
campaign_versions

events_after_initial_end = ch("""
SELECT
    count() AS events,
    round(sum(toFloat64(revenue_usd)), 6) AS total_revenue_usd
FROM raw.ad_events
WHERE campaign_id = 1017
  AND toDate(event_timestamp) > toDate('2026-03-05')
  AND toDate(event_timestamp) <= toDate('2026-03-20')
""")
events_after_initial_end

Finding 4: Campaign 1017 flight mismatch
WHAT: Initial export: status completed, end 2026-03-05. Redelivery (FINAL): active, end 2026-03-20. 224 events between 2026-03-06 and 2026-03-20 ($0.48 revenue) are in-flight under redelivery only.
WHY IT MATTERS: Flight filters and budget pacing depend on dimension version; wrong version misclassifies valid traffic.
HOW HANDLED: Dagster loads redelivery overlay; dbt uses FINAL (SCD Type 1 latest). Production: SCD2 if audit trail required.


,events,total_revenue_usd
0,224,0.481987


## Finding 5 — Publisher timezone shifts daily revenue (Publisher 11)

**Hypothesis:** UTC calendar day mis-allocates revenue vs publisher-local reporting.

In [8]:
show_finding(
    "Finding 5: UTC vs publisher-local event day",
    what="Publisher 11 (America/Los_Angeles): 2026-03-19 revenue = $5.18 (UTC bucket) vs $181.46 (LA bucket). "
         "Total publisher revenue unchanged; ~$418 reallocated across days when rebucketing.",
    why="Publisher-facing daily reports must use publisher timezone; UTC breaks evening traffic allocation.",
    handled="fct_ad_events_daily uses event_date in publisher timezone (see stg_ad_events / DESIGN §2b).",
)

tz_pub11_mar19 = ch("""
SELECT
    round(sumIf(toFloat64(revenue_usd), toDate(event_timestamp) = '2026-03-19'), 6) AS revenue_utc_mar19,
    round(sumIf(
        toFloat64(revenue_usd),
        toDate(toTimeZone(event_timestamp, 'America/Los_Angeles')) = '2026-03-19'
    ), 6) AS revenue_la_mar19
FROM raw.ad_events
WHERE publisher_id = 11 AND revenue_usd IS NOT NULL
""")
tz_pub11_mar19

Finding 5: UTC vs publisher-local event day
WHAT: Publisher 11 (America/Los_Angeles): 2026-03-19 revenue = $5.18 (UTC bucket) vs $181.46 (LA bucket). Total publisher revenue unchanged; ~$418 reallocated across days when rebucketing.
WHY IT MATTERS: Publisher-facing daily reports must use publisher timezone; UTC breaks evening traffic allocation.
HOW HANDLED: fct_ad_events_daily uses event_date in publisher timezone (see stg_ad_events / DESIGN §2b).


,revenue_utc_mar19,revenue_la_mar19
0,5.179692,181.460081


## Finding 6 — Publisher / campaign attribute changes (redelivery)

**Hypothesis:** Operational dimensions change between deliveries; reporting attributes must use latest overlay.

In [9]:
show_finding(
    "Finding 6: Dimension attribute drift",
    what="Publishers: id 4 account_manager Priya Singh → Hannah Reid; id 12 category Lifestyle → Entertainment. "
         "Campaign 1003 budget $320,000 → $360,000. Campaign 1017 per Finding 4.",
    why="Account management and categorisation reports shift when redelivery lands.",
    handled="Ingestion inserts initial then redelivery; dbt reads FINAL / Type 1 dims.",
)

dim_drift = ch("""
SELECT
    publisher_id,
    groupUniqArray(account_manager) AS account_managers,
    groupUniqArray(publisher_category) AS categories
FROM raw.publishers
GROUP BY publisher_id
HAVING length(account_managers) > 1 OR length(categories) > 1
ORDER BY publisher_id
""")
dim_drift

campaign_budget = ch("""
SELECT
    campaign_id,
    groupUniqArray(toString(campaign_budget_usd)) AS budgets
FROM raw.campaigns
GROUP BY campaign_id
HAVING length(budgets) > 1
ORDER BY campaign_id
""")
campaign_budget

Finding 6: Dimension attribute drift
WHAT: Publishers: id 4 account_manager Priya Singh → Hannah Reid; id 12 category Lifestyle → Entertainment. Campaign 1003 budget $320,000 → $360,000. Campaign 1017 per Finding 4.
WHY IT MATTERS: Account management and categorisation reports shift when redelivery lands.
HOW HANDLED: Ingestion inserts initial then redelivery; dbt reads FINAL / Type 1 dims.


,campaign_id,budgets
0,1003,"[360000, 320000]"


## Finding 7 — Site domain ≠ publisher primary domain

**Hypothesis:** Cross-domain serving is valid in programmatic but should be visible in integrity checks.

In [10]:
show_finding(
    "Finding 7: site_domain vs primary_domain mismatch",
    what="657 events ($1.50 revenue) where site_domain ≠ publishers.primary_domain (FINAL join).",
    why="Small $ impact but signals syndication / network serving worth monitoring.",
    handled="No filter in mart; flag in QA dashboard in production.",
)

domain_mismatch = ch("""
SELECT
    count() AS mismatch_events,
    round(sum(toFloat64(e.revenue_usd)), 6) AS revenue_usd
FROM raw.ad_events AS e
INNER JOIN (
    SELECT publisher_id, primary_domain FROM raw.publishers FINAL
) AS p ON e.publisher_id = p.publisher_id
WHERE e.site_domain != '' AND e.site_domain != p.primary_domain
""")
domain_mismatch

Finding 7: site_domain vs primary_domain mismatch
WHAT: 657 events ($1.50 revenue) where site_domain ≠ publishers.primary_domain (FINAL join).
WHY IT MATTERS: Small $ impact but signals syndication / network serving worth monitoring.
HOW HANDLED: No filter in mart; flag in QA dashboard in production.


,mismatch_events,revenue_usd
0,657,1.498876
